In [5]:
"""
TOP1 高斯过程模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整残差分析程序 v2（修复版）
================================================================
修复：qq_confidence_band 返回7个值，所有调用处统一改为接收7个变量
  - 三处外部调用：tq, sq, cil, ciu, fit, slope, intercept
  - draw_qq_with_ci 内部调用：同步修正，sw_p 单独用 shapiro() 计算

v2新增功能（完整保留）：
  图5: Q-Q图完整增强版（2×3，6个子图）
    (0,0) 训练集Q-Q + 95%置信带
    (0,1) 测试集Q-Q + 95%置信带
    (0,2) 全数据Q-Q + 95%置信带
    (1,0) 训练集标准化残差Q-Q（对比N(0,1)）
    (1,1) 测试集标准化残差Q-Q（对比N(0,1)）
    (1,2) 全数据标准化残差Q-Q（对比N(0,1)）

Excel共29个工作表（原26 + 新增3）：
  Sheet 27: QQ_All_Data         → 全数据Q-Q原始数据
  Sheet 28: QQ_Confidence_Band  → 置信带数据（训练/测试/全数据）
  Sheet 29: QQ_Standardized     → 标准化残差Q-Q数据（训练/测试/全数据）

输出目录: D:\\博二上\\模型分析可视化\\残差分析2
"""

import os
import warnings
import numpy as np
import pandas as pd
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from scipy.stats import (pearsonr, shapiro, kstest, anderson,
                         jarque_bera, gaussian_kde)
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

print("=" * 80)
print("TOP1 高斯过程模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整残差分析程序 v2（修复版）")
print("修复: qq_confidence_band 返回值解包对齐（7个变量）")
print("=" * 80)

# ============================================================
# 路径配置
# ============================================================
PKL_FILE = (
    r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
    r"\top30_models\model_objects"
    r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
)
DATA_FILE = (
    r"D:\博一下\第一章主动学习"
    r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
)
OUTPUT_DIR = r"D:\博二上\模型分析可视化\残差分析2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 2023
TEST_SIZE    = 0.2
TARGET_COL   = 'KQ'

# ============================================================
# GP 专用预测函数
# ============================================================
def gp_predict_scaled(model, scaler, X_raw):
    return model.predict(scaler.transform(X_raw))

def gp_predict_with_std(model, scaler, X_raw):
    try:
        y_mean, y_std = model.predict(scaler.transform(X_raw), return_std=True)
        return y_mean, y_std
    except Exception:
        y_mean = model.predict(scaler.transform(X_raw))
        return y_mean, np.zeros_like(y_mean)

def calc_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def create_stratify_bins(y, n_bins=5):
    percentiles   = np.linspace(0, 100, n_bins + 1)
    bin_edges     = np.percentile(y, percentiles)
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
    return np.digitize(y, bin_edges) - 1

# ============================================================
# ★ Q-Q置信带计算工具函数（修复版）
# ============================================================
def qq_confidence_band(residuals, ci=0.95, n_boot=1000):
    """
    通过Bootstrap计算Q-Q图的置信带。
    返回7个值：theoretical_q, sample_q, ci_lower, ci_upper,
               fit_line, slope, intercept
    """
    n       = len(residuals)
    res_std = (residuals - residuals.mean()) / residuals.std()

    qq_result        = stats.probplot(res_std, dist='norm')
    theoretical_q    = qq_result[0][0]
    sample_q         = qq_result[0][1]
    slope            = qq_result[1][0]
    intercept        = qq_result[1][1]
    fit_line         = theoretical_q * slope + intercept

    boot_quantiles = []
    for _ in range(n_boot):
        boot_sample = np.random.choice(res_std, size=n, replace=True)
        boot_sorted = np.sort(boot_sample)
        boot_quantiles.append(boot_sorted)

    boot_arr = np.array(boot_quantiles)
    alpha    = 1 - ci
    ci_lower = np.percentile(boot_arr, alpha / 2 * 100,       axis=0)
    ci_upper = np.percentile(boot_arr, (1 - alpha / 2) * 100, axis=0)

    # 返回7个值
    return theoretical_q, sample_q, ci_lower, ci_upper, fit_line, slope, intercept


def draw_qq_with_ci(ax, residuals, title, color='steelblue',
                    ci=0.95, n_boot=500, label_suffix=''):
    """
    在指定ax上绘制带置信带的Q-Q图。
    ★ 修复：接收qq_confidence_band的7个返回值，sw_p单独计算
    """
    # ★ 修复：解包7个变量
    theoretical_q, sample_q, ci_lower, ci_upper, fit_line, slope, intercept = \
        qq_confidence_band(residuals, ci=ci, n_boot=n_boot)

    # sw_p 单独用 shapiro 计算（不从qq_confidence_band获取）
    sw_stat, sw_p = shapiro(residuals)

    # 置信带（阴影）
    ax.fill_between(theoretical_q, ci_lower, ci_upper,
                    alpha=0.25, color=color,
                    label=f'{int(ci*100)}% Confidence Band')

    # 对角参考线
    ax.plot(theoretical_q, fit_line,
            'r-', lw=2, zorder=3, label='Reference Line')

    # Q-Q散点
    ax.scatter(theoretical_q, sample_q,
               color=color, s=35, alpha=0.85,
               edgecolors='black', linewidth=0.4,
               zorder=4, label=f'Q-Q Points{label_suffix}')

    ax.set_xlabel('Theoretical Quantiles', fontsize=11, fontweight='bold')
    ax.set_ylabel('Sample Quantiles',      fontsize=11, fontweight='bold')
    ax.set_title(f'{title}\nShapiro-Wilk p={sw_p:.4f}  '
                 f'{"✓ Normal" if sw_p > 0.05 else "⚠ Non-normal"}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

    return theoretical_q, sample_q, ci_lower, ci_upper, fit_line, sw_p


def draw_standardized_qq(ax, residuals, title, color='steelblue'):
    """
    绘制标准化残差Q-Q图（残差标准化后与N(0,1)对比）。
    """
    res_std = (residuals - residuals.mean()) / residuals.std()
    n       = len(res_std)

    theoretical_q   = stats.norm.ppf(np.linspace(0.5/n, 1 - 0.5/n, n))
    sample_q_sorted = np.sort(res_std)

    ax.scatter(theoretical_q, sample_q_sorted,
               color=color, s=35, alpha=0.85,
               edgecolors='black', linewidth=0.4,
               zorder=3, label='Standardized Residuals')

    min_val = min(theoretical_q.min(), sample_q_sorted.min())
    max_val = max(theoretical_q.max(), sample_q_sorted.max())
    ax.plot([min_val, max_val], [min_val, max_val],
            'r--', lw=2, zorder=2, label='N(0,1) Ideal')

    for sigma, ls, alpha_v in [(1, ':', 0.6), (2, ':', 0.4)]:
        ax.axhline( sigma, color='gray', ls=ls, lw=1.2, alpha=alpha_v)
        ax.axhline(-sigma, color='gray', ls=ls, lw=1.2, alpha=alpha_v)
        ax.text(max_val * 0.95,  sigma + 0.05, f'+{sigma}σ',
                fontsize=8, color='gray', ha='right')
        ax.text(max_val * 0.95, -sigma + 0.05, f'-{sigma}σ',
                fontsize=8, color='gray', ha='right')

    sw_stat, sw_p = shapiro(residuals)
    skew_val      = float(pd.Series(res_std).skew())
    kurt_val      = float(pd.Series(res_std).kurt())

    ax.text(0.03, 0.97,
            f'n={n}\nSkew={skew_val:.3f}\nKurt={kurt_val:.3f}',
            transform=ax.transAxes, fontsize=9,
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))

    ax.set_xlabel('Theoretical Quantiles N(0,1)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Standardized Residuals',        fontsize=11, fontweight='bold')
    ax.set_title(f'{title}\nShapiro-Wilk p={sw_p:.4f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

    return theoretical_q, sample_q_sorted, sw_p

# ============================================================
# 步骤 1: 加载模型和数据
# ============================================================
print("\n[1/5] 加载 TOP1 高斯过程模型和数据...")

with open(PKL_FILE, 'rb') as f:
    model_data = pickle.load(f)

gp_model      = model_data['model']
scaler        = model_data['scaler']
feature_names = model_data['features']

print(f"  ✓ 模型名称: {model_data['model_name']}")
print(f"  ✓ 特征列表: {feature_names}")
print(f"  ✓ 原始 Test R²:  {model_data.get('test_r2_true', 'N/A')}")
print(f"  ✓ 原始 Test MAE: {model_data.get('test_mae_true', 'N/A')}")

df       = pd.read_excel(DATA_FILE)
df_valid = df[feature_names + [TARGET_COL]].dropna()
df_valid = df_valid[df_valid[TARGET_COL] > 0].reset_index(drop=True)
X_all    = df_valid[feature_names].values
y_all    = df_valid[TARGET_COL].values

print(f"  ✓ 原始数据: {df.shape}  有效样本: {len(df_valid)}")
print(f"  ✓ 目标变量范围: [{y_all.min():.3f}, {y_all.max():.3f}]")

# ============================================================
# 步骤 2: 数据划分
# ============================================================
print("\n[2/5] 数据划分...")

strat   = create_stratify_bins(y_all)
idx_all = np.arange(len(y_all))
train_idx, test_idx, _, _ = train_test_split(
    idx_all, idx_all,
    test_size=TEST_SIZE, stratify=strat,
    random_state=RANDOM_STATE)
X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
print(f"  ✓ 训练集: {len(y_train)} 样本  测试集: {len(y_test)} 样本")

# ============================================================
# 步骤 3: 模型推理
# ============================================================
print("\n[3/5] 模型推理...")

train_pred, train_std = gp_predict_with_std(gp_model, scaler, X_train)
test_pred,  test_std  = gp_predict_with_std(gp_model, scaler, X_test)
all_pred,   all_std   = gp_predict_with_std(gp_model, scaler, X_all)

train_r2   = r2_score(y_train, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_mae  = mean_absolute_error(y_train, train_pred)
train_mape = calc_mape(y_train, train_pred)
test_r2    = r2_score(y_test,  test_pred)
test_rmse  = np.sqrt(mean_squared_error(y_test,  test_pred))
test_mae   = mean_absolute_error(y_test,  test_pred)
test_mape  = calc_mape(y_test,  test_pred)

print(f"  训练集: R²={train_r2:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}")
print(f"  测试集: R²={test_r2:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}")

# ============================================================
# 步骤 4: 残差计算与统计检验
# ============================================================
print("\n[4/5] 残差分析与统计检验...")

train_residuals = y_train - train_pred
test_residuals  = y_test  - test_pred
all_residuals   = y_all   - all_pred

train_res_std = (train_residuals - train_residuals.mean()) / train_residuals.std()
test_res_std  = (test_residuals  - test_residuals.mean())  / test_residuals.std()
all_res_std   = (all_residuals   - all_residuals.mean())   / all_residuals.std()

shapiro_stat, shapiro_p   = shapiro(all_residuals)
ks_stat,      ks_p        = kstest(all_res_std, 'norm')
anderson_res               = anderson(all_residuals, dist='norm')
jb_stat,      jb_p        = jarque_bera(all_residuals)
dw_stat                    = durbin_watson(all_residuals)
X_const                    = np.column_stack([np.ones(len(all_pred)), all_pred])
bp_stat, bp_p, _, _        = het_breuschpagan(all_residuals, X_const)
corr_all,     p_corr       = pearsonr(all_pred, all_residuals)

print(f"  Shapiro-Wilk: p={shapiro_p:.4f}  {'✓' if shapiro_p>0.05 else '⚠'}")
print(f"  K-S Test:     p={ks_p:.4f}  {'✓' if ks_p>0.05 else '⚠'}")
print(f"  Durbin-Watson: {dw_stat:.4f}  {'✓' if 1.5<dw_stat<2.5 else '⚠'}")
print(f"  Breusch-Pagan: p={bp_p:.4f}  {'✓' if bp_p>0.05 else '⚠'}")

passed = sum([
    shapiro_p > 0.05, ks_p > 0.05, jb_p > 0.05,
    anderson_res.statistic < anderson_res.critical_values[2],
    1.5 < dw_stat < 2.5, bp_p > 0.05, abs(corr_all) < 0.3,
])
print(f"  通过检验: {passed}/7 ({passed/7*100:.1f}%)")

within_1      = np.mean(np.abs(all_residuals) <= all_std) * 100
within_2      = np.mean(np.abs(all_residuals) <= 2 * all_std) * 100
norm_resid_gp = all_residuals / (all_std + 1e-10)

z_het       = np.polyfit(all_pred, np.abs(all_residuals), 1)
trend_x_het = np.sort(all_pred)

max_lags_acf  = min(20, len(all_residuals) // 2 - 1)
max_lags_pacf = min(15, len(all_residuals) // 2 - 2)
window        = min(3, len(all_residuals) // 3)
roll_mean     = pd.Series(all_residuals).rolling(window).mean()
roll_std      = pd.Series(all_residuals).rolling(window).std()

if max_lags_acf > 0:
    acf_vals = acf(all_residuals, nlags=max_lags_acf, fft=False)
    acf_df   = pd.DataFrame({'Lag': range(len(acf_vals)), 'ACF': acf_vals})
else:
    acf_df = pd.DataFrame({'Note': ['Sample too small']})

if max_lags_pacf > 0:
    try:
        pacf_vals = pacf(all_residuals, nlags=max_lags_pacf)
        pacf_df   = pd.DataFrame({'Lag': range(len(pacf_vals)), 'PACF': pacf_vals})
    except Exception:
        pacf_df = pd.DataFrame({'Note': ['PACF failed']})
else:
    pacf_df = pd.DataFrame({'Note': ['Sample too small']})

# ============================================================
# 预先计算Q-Q置信带数据（用于Excel保存）
# ★ 修复：接收7个返回值
# ============================================================
print("\n  ★ 计算Q-Q置信带数据（Bootstrap n=500）...")

CI_LEVEL  = 0.95
N_BOOT_QQ = 500

# ★ 修复：训练集 - 接收7个变量
tq_tr, sq_tr, cil_tr, ciu_tr, fit_tr, slope_tr, intercept_tr = \
    qq_confidence_band(train_residuals, ci=CI_LEVEL, n_boot=N_BOOT_QQ)

# ★ 修复：测试集 - 接收7个变量
tq_te, sq_te, cil_te, ciu_te, fit_te, slope_te, intercept_te = \
    qq_confidence_band(test_residuals, ci=CI_LEVEL, n_boot=N_BOOT_QQ)

# ★ 修复：全数据 - 接收7个变量
tq_all, sq_all, cil_all, ciu_all, fit_all, slope_all, intercept_all = \
    qq_confidence_band(all_residuals, ci=CI_LEVEL, n_boot=N_BOOT_QQ)

# Shapiro-Wilk p值单独计算
sw_p_tr  = shapiro(train_residuals)[1]
sw_p_te  = shapiro(test_residuals)[1]
sw_p_all = shapiro(all_residuals)[1]

print(f"  ✓ 置信带计算完成（训练: n={len(train_residuals)}, "
      f"测试: n={len(test_residuals)}, 全: n={len(all_residuals)}）")

# ============================================================
# 保存 Excel（原26个 + 新增3个 = 29个工作表）
# ============================================================
print("\n  保存 Excel 数据文件（29个工作表）...")

def hist_data(res, bins):
    counts, edges = np.histogram(res, bins=bins, density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    return pd.DataFrame({'Bin_Center': centers, 'Density': counts,
                         'Bin_Left': edges[:-1], 'Bin_Right': edges[1:]})

def normal_fit(res, n=100):
    mu, sigma = res.mean(), res.std()
    x = np.linspace(res.min(), res.max(), n)
    return pd.DataFrame({'X': x, 'Normal_PDF': stats.norm.pdf(x, mu, sigma)})

def kde_data(res, n=100):
    kde = gaussian_kde(res)
    x   = np.linspace(res.min(), res.max(), n)
    return pd.DataFrame({'X': x, 'KDE_Density': kde(x)})

def qq_data(res_std):
    qq = stats.probplot(res_std, dist='norm')
    slope_v, intercept_v = qq[1][0], qq[1][1]
    return pd.DataFrame({
        'Theoretical_Quantiles': qq[0][0],
        'Sample_Quantiles':      qq[0][1],
        'Fit_Line_Y':            qq[0][0] * slope_v + intercept_v,
    })

def box_stats(res, name):
    return {
        'Dataset': name, 'Min': res.min(),
        'Q1':  np.percentile(res, 25), 'Median': np.median(res),
        'Q3':  np.percentile(res, 75), 'Max': res.max(),
        'Mean': res.mean(), 'Std': res.std(),
        'IQR': np.percentile(res, 75) - np.percentile(res, 25),
    }

def desc_stats(res):
    return {
        'Count': len(res), 'Mean': res.mean(), 'Std': res.std(),
        'Min': res.min(), 'Max': res.max(),
        'Range': res.max() - res.min(),
        'Skewness': stats.skew(res), 'Kurtosis': stats.kurtosis(res),
        'Q1': np.percentile(res, 25), 'Median': np.median(res),
        'Q3': np.percentile(res, 75),
        'IQR': np.percentile(res, 75) - np.percentile(res, 25),
        'CI_Lower_95%': np.percentile(res, 2.5),
        'CI_Upper_95%': np.percentile(res, 97.5),
    }

# 原有26个Sheet的数据
scatter_df = pd.DataFrame({
    'Actual_KQ':    np.concatenate([y_train, y_test]),
    'Predicted_KQ': np.concatenate([train_pred, test_pred]),
    'Pred_Std_GP':  np.concatenate([train_std,  test_std]),
    'Dataset':      ['Train']*len(y_train) + ['Test']*len(y_test),
})
residual_plot_df = pd.DataFrame({
    'Predicted_KQ': np.concatenate([train_pred, test_pred]),
    'Residuals':    np.concatenate([train_residuals, test_residuals]),
    'Pred_Std_GP':  np.concatenate([train_std, test_std]),
    'Dataset':      ['Train']*len(y_train) + ['Test']*len(y_test),
})
detailed_df = pd.DataFrame({
    'Sample_Index':          range(len(y_all)),
    'Actual_KQ':             y_all, 'Predicted_KQ': all_pred,
    'GP_Pred_Std':           all_std, 'Residual': all_residuals,
    'Abs_Residual':          np.abs(all_residuals),
    'Standardized_Residual': all_res_std,
    'Squared_Residual':      all_residuals**2,
    'Percentage_Error_%':    (all_residuals/y_all)*100,
    'APE_%':                 np.abs(all_residuals/y_all)*100,
    'Normalized_Resid_GP':   norm_resid_gp,
    'Within_1_GP_Std':       (np.abs(all_residuals)<=all_std).astype(int),
    'Within_2_GP_Std':       (np.abs(all_residuals)<=2*all_std).astype(int),
})
hist_train_df   = hist_data(train_residuals, 12)
hist_test_df    = hist_data(test_residuals,   8)
hist_all_df     = hist_data(all_residuals,   15)
normal_train_df = normal_fit(train_residuals)
normal_test_df  = normal_fit(test_residuals)
normal_all_df   = normal_fit(all_residuals)
kde_train_df    = kde_data(train_residuals)
kde_test_df     = kde_data(test_residuals)
qq_train_df     = qq_data(train_res_std)
qq_test_df      = qq_data(test_res_std)
qq_all_df_orig  = qq_data(all_res_std)
box_df          = pd.DataFrame([
    box_stats(train_residuals,'Train'),
    box_stats(test_residuals, 'Test'),
    box_stats(all_residuals,  'All'),
])
std_res_df = pd.DataFrame({
    'Sample_Index': range(len(all_res_std)),
    'Standardized_Residuals': all_res_std,
    'Plus_2_Sigma': 2, 'Minus_2_Sigma': -2,
    'Plus_3_Sigma': 3, 'Minus_3_Sigma': -3,
})
hetero_data_df = pd.DataFrame({
    'Predicted_KQ': all_pred, 'Abs_Residuals': np.abs(all_residuals),
    'GP_Pred_Std':  all_std,
})
hetero_trend_df = pd.DataFrame({
    'Predicted_KQ': trend_x_het,
    'Trend_Line':   z_het[0]*trend_x_het+z_het[1],
    'Slope': z_het[0], 'Intercept': z_het[1],
})
moving_df = pd.DataFrame({
    'Sample_Index': range(len(all_residuals)),
    'Residuals': all_residuals, 'Rolling_Mean': roll_mean,
    'Rolling_Std': roll_std,
    'Upper_Band': roll_mean+roll_std, 'Lower_Band': roll_mean-roll_std,
})
cusum_df = pd.DataFrame({
    'Sample_Index': range(len(all_residuals)),
    'CUSUM': np.cumsum(all_residuals), 'Zero_Line': 0,
})
stats_summary_df = pd.DataFrame({
    'Test_Name': [
        'Shapiro-Wilk (Normality)', 'Kolmogorov-Smirnov (Normality)',
        'Anderson-Darling (Normality)', 'Jarque-Bera (Normality)',
        'Durbin-Watson (Independence)', 'Breusch-Pagan (Homoscedasticity)',
        'Correlation (Pred vs Residual)',
    ],
    'Statistic': [shapiro_stat, ks_stat, anderson_res.statistic,
                  jb_stat, dw_stat, bp_stat, corr_all],
    'P_Value':   [shapiro_p, ks_p, np.nan, jb_p, np.nan, bp_p, p_corr],
    'Interpretation': [
        'Pass' if shapiro_p>0.05 else 'Attention',
        'Pass' if ks_p>0.05 else 'Attention',
        'Pass' if anderson_res.statistic<anderson_res.critical_values[2] else 'Attention',
        'Pass' if jb_p>0.05 else 'Attention',
        'Pass' if 1.5<dw_stat<2.5 else 'Attention',
        'Pass' if bp_p>0.05 else 'Attention',
        'Pass' if abs(corr_all)<0.3 else 'Attention',
    ],
})
d_tr  = desc_stats(train_residuals)
d_te  = desc_stats(test_residuals)
d_all = desc_stats(all_residuals)
desc_df = pd.DataFrame({
    'Metric': list(d_tr.keys()),
    'Training_Set': list(d_tr.values()),
    'Test_Set':     list(d_te.values()),
    'All_Data':     list(d_all.values()),
})
perf_df = pd.DataFrame({
    'Dataset': ['Training','Test'],
    'R2':      [train_r2,  test_r2],
    'RMSE':    [train_rmse,test_rmse],
    'MAE':     [train_mae, test_mae],
    'MAPE_%':  [train_mape,test_mape],
    'Samples': [len(y_train), len(y_test)],
    'Features':[', '.join(feature_names)]*2,
})
gp_uncertainty_df = pd.DataFrame({
    'Dataset':    ['Train']*len(train_std) + ['Test']*len(test_std),
    'Predicted_KQ': np.concatenate([train_pred, test_pred]),
    'Actual_KQ':    np.concatenate([y_train, y_test]),
    'GP_Pred_Std':  np.concatenate([train_std, test_std]),
    'Residual':     np.concatenate([train_residuals, test_residuals]),
    'Normalized_Resid_GP': np.concatenate([
        train_residuals/(train_std+1e-10),
        test_residuals/ (test_std +1e-10)]),
    'Within_1std': (np.abs(np.concatenate([train_residuals,test_residuals])) <=
                    np.concatenate([train_std,test_std])).astype(int),
    'Within_2std': (np.abs(np.concatenate([train_residuals,test_residuals])) <=
                    2*np.concatenate([train_std,test_std])).astype(int),
})

# ── v2新增：Sheet 27-29 ──────────────────────────────────────

# Sheet 27: 全数据Q-Q原始数据
qq_all_enhanced_df = pd.DataFrame({
    'Theoretical_Quantiles': tq_all,
    'Sample_Quantiles':      sq_all,
    'Fit_Line_Y':            fit_all,
    'CI_Lower':              cil_all,
    'CI_Upper':              ciu_all,
    'Dataset':               'All',
    'n_samples':             len(all_residuals),
    'Shapiro_p':             sw_p_all,
    'Slope':                 slope_all,
    'Intercept':             intercept_all,
})

# Sheet 28: 置信带宽表（训练/测试/全数据）
n_max = max(len(tq_tr), len(tq_te), len(tq_all))
def pad(arr, n):
    padded = np.full(n, np.nan)
    padded[:len(arr)] = arr
    return padded

qq_ci_df = pd.DataFrame({
    'Train_Theoretical_Q': pad(tq_tr,  n_max),
    'Train_Sample_Q':      pad(sq_tr,  n_max),
    'Train_CI_Lower':      pad(cil_tr, n_max),
    'Train_CI_Upper':      pad(ciu_tr, n_max),
    'Train_Fit_Line':      pad(fit_tr, n_max),
    'Test_Theoretical_Q':  pad(tq_te,  n_max),
    'Test_Sample_Q':       pad(sq_te,  n_max),
    'Test_CI_Lower':       pad(cil_te, n_max),
    'Test_CI_Upper':       pad(ciu_te, n_max),
    'Test_Fit_Line':       pad(fit_te, n_max),
    'All_Theoretical_Q':   pad(tq_all,  n_max),
    'All_Sample_Q':        pad(sq_all,  n_max),
    'All_CI_Lower':        pad(cil_all, n_max),
    'All_CI_Upper':        pad(ciu_all, n_max),
    'All_Fit_Line':        pad(fit_all, n_max),
})

# Sheet 29: 标准化残差Q-Q数据
def std_qq_data(residuals, label):
    res_std = (residuals - residuals.mean()) / residuals.std()
    n       = len(res_std)
    tq      = stats.norm.ppf(np.linspace(0.5/n, 1-0.5/n, n))
    sq      = np.sort(res_std)
    sw_p_val = shapiro(residuals)[1]
    return pd.DataFrame({
        f'{label}_Theoretical_Q':  tq,
        f'{label}_Standardized_Q': sq,
        f'{label}_Ideal_Line_Y':   tq,
        f'{label}_Shapiro_p':      sw_p_val,
        f'{label}_Skewness':       float(pd.Series(sq).skew()),
        f'{label}_Kurtosis':       float(pd.Series(sq).kurt()),
    })

df_std_tr  = std_qq_data(train_residuals, 'Train')
df_std_te  = std_qq_data(test_residuals,  'Test')
df_std_all = std_qq_data(all_residuals,   'All')

qq_std_df = pd.concat([
    df_std_tr.reset_index(drop=True),
    df_std_te.reset_index(drop=True),
    df_std_all.reset_index(drop=True),
], axis=1)

excel_path = os.path.join(OUTPUT_DIR, 'GP_Matern_Residual_Analysis_Data.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    scatter_df.to_excel(        writer, sheet_name='01_Scatter_Actual_vs_Pred',  index=False)
    residual_plot_df.to_excel(  writer, sheet_name='02_Residual_Plot',            index=False)
    detailed_df.to_excel(       writer, sheet_name='03_Detailed_Residuals',       index=False)
    hist_train_df.to_excel(     writer, sheet_name='04_Histogram_Train',          index=False)
    hist_test_df.to_excel(      writer, sheet_name='05_Histogram_Test',           index=False)
    hist_all_df.to_excel(       writer, sheet_name='06_Histogram_All',            index=False)
    normal_train_df.to_excel(   writer, sheet_name='07_Normal_Fit_Train',         index=False)
    normal_test_df.to_excel(    writer, sheet_name='08_Normal_Fit_Test',          index=False)
    normal_all_df.to_excel(     writer, sheet_name='09_Normal_Fit_All',           index=False)
    kde_train_df.to_excel(      writer, sheet_name='10_KDE_Train',                index=False)
    kde_test_df.to_excel(       writer, sheet_name='11_KDE_Test',                 index=False)
    qq_train_df.to_excel(       writer, sheet_name='12_QQ_Plot_Train',            index=False)
    qq_test_df.to_excel(        writer, sheet_name='13_QQ_Plot_Test',             index=False)
    qq_all_df_orig.to_excel(    writer, sheet_name='14_QQ_Plot_All',              index=False)
    box_df.to_excel(            writer, sheet_name='15_Boxplot_Statistics',       index=False)
    std_res_df.to_excel(        writer, sheet_name='16_Standardized_Residuals',   index=False)
    hetero_data_df.to_excel(    writer, sheet_name='17_Heteroscedasticity_Data',  index=False)
    hetero_trend_df.to_excel(   writer, sheet_name='18_Heteroscedasticity_Trend', index=False)
    acf_df.to_excel(            writer, sheet_name='19_ACF_Data',                 index=False)
    pacf_df.to_excel(           writer, sheet_name='20_PACF_Data',                index=False)
    moving_df.to_excel(         writer, sheet_name='21_Moving_Average',           index=False)
    cusum_df.to_excel(          writer, sheet_name='22_CUSUM',                    index=False)
    stats_summary_df.to_excel(  writer, sheet_name='23_Statistical_Tests',        index=False)
    desc_df.to_excel(           writer, sheet_name='24_Descriptive_Statistics',   index=False)
    perf_df.to_excel(           writer, sheet_name='25_Model_Performance',        index=False)
    gp_uncertainty_df.to_excel( writer, sheet_name='26_GP_Uncertainty_Analysis',  index=False)
    # v2新增3个Sheet
    qq_all_enhanced_df.to_excel(writer, sheet_name='27_QQ_All_Data',             index=False)
    qq_ci_df.to_excel(          writer, sheet_name='28_QQ_Confidence_Band',      index=False)
    qq_std_df.to_excel(         writer, sheet_name='29_QQ_Standardized',         index=False)

print(f"  ✓ Excel 已保存（29个工作表）: {os.path.basename(excel_path)}")

# ============================================================
# 步骤 5: 可视化（原4张 + 新增图5）
# ============================================================
print("\n[5/5] 生成可视化图片（原4张 + 新增图5）...")
plt.style.use('seaborn-v0_8-darkgrid')

# ==============================================================
# 图1：综合残差分析（3×3）
# ==============================================================
fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.32)

ax = fig.add_subplot(gs[0, 0])
ax.scatter(train_pred, train_residuals, alpha=0.6, s=60,
           color='steelblue', edgecolors='black', linewidth=0.5)
ax.axhline(0, color='red', lw=2, ls='--')
ax.axhline( train_residuals.std()*2, color='orange', lw=1.5, ls=':', label='+2σ')
ax.axhline(-train_residuals.std()*2, color='orange', lw=1.5, ls=':', label='-2σ')
ax.set_xlabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuals', fontsize=11, fontweight='bold')
ax.set_title(f'Training Set: Residuals vs Predicted\nR²={train_r2:.4f}, RMSE={train_rmse:.4f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[0, 1])
ax.scatter(test_pred, test_residuals, alpha=0.6, s=60,
           color='orange', edgecolors='black', linewidth=0.5)
ax.axhline(0, color='red', lw=2, ls='--')
ax.axhline( test_residuals.std()*2, color='blue', lw=1.5, ls=':', label='+2σ')
ax.axhline(-test_residuals.std()*2, color='blue', lw=1.5, ls=':', label='-2σ')
ax.set_xlabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuals', fontsize=11, fontweight='bold')
ax.set_title(f'Test Set: Residuals vs Predicted\nR²={test_r2:.4f}, RMSE={test_rmse:.4f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[0, 2])
ax.scatter(all_pred, all_residuals, alpha=0.6, s=60,
           color='green', edgecolors='black', linewidth=0.5)
ax.axhline(0, color='red', lw=2, ls='--')
ax.axhline( all_residuals.std()*2, color='purple', lw=1.5, ls=':', label='+2σ')
ax.axhline(-all_residuals.std()*2, color='purple', lw=1.5, ls=':', label='-2σ')
ax.set_xlabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuals', fontsize=11, fontweight='bold')
ax.set_title('All Data: Residuals vs Predicted', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1, 0])
mu_a, sig_a = all_residuals.mean(), all_residuals.std()
ax.hist(all_residuals, bins=15, density=True, alpha=0.7,
        color='skyblue', edgecolor='black', label='Residuals')
x_n = np.linspace(all_residuals.min(), all_residuals.max(), 100)
ax.plot(x_n, stats.norm.pdf(x_n, mu_a, sig_a), 'r-', lw=2, label='Normal Fit')
ax.set_xlabel('Residuals', fontsize=11, fontweight='bold')
ax.set_ylabel('Density',   fontsize=11, fontweight='bold')
ax.set_title(f'Residuals Distribution\nμ={mu_a:.4f}, σ={sig_a:.4f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1, 1])
stats.probplot(all_residuals, dist='norm', plot=ax)
ax.set_xlabel('Theoretical Quantiles', fontsize=11, fontweight='bold')
ax.set_ylabel('Sample Quantiles',      fontsize=11, fontweight='bold')
ax.set_title('Q-Q Plot (Normality Test)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1, 2])
bp_plot = ax.boxplot([train_residuals, test_residuals, all_residuals],
                     labels=['Train','Test','All'],
                     patch_artist=True, showmeans=True)
for patch, c in zip(bp_plot['boxes'], ['lightblue','lightsalmon','lightgreen']):
    patch.set_facecolor(c)
ax.axhline(0, color='red', lw=2, ls='--')
ax.set_ylabel('Residuals', fontsize=11, fontweight='bold')
ax.set_title('Residuals Distribution Comparison', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = fig.add_subplot(gs[2, 0])
ax.scatter(range(len(all_res_std)), all_res_std,
           alpha=0.6, s=40, color='teal', edgecolors='black', linewidth=0.4)
ax.axhline(0,  color='red',    lw=2,   ls='--')
ax.axhline( 2, color='orange', lw=1.5, ls=':', label='±2σ')
ax.axhline(-2, color='orange', lw=1.5, ls=':')
ax.axhline( 3, color='red',    lw=1.5, ls=':', label='±3σ')
ax.axhline(-3, color='red',    lw=1.5, ls=':')
ax.set_xlabel('Sample Index',           fontsize=11, fontweight='bold')
ax.set_ylabel('Standardized Residuals', fontsize=11, fontweight='bold')
ax.set_title('Standardized Residuals vs Index', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[2, 1])
ax.scatter(all_pred, np.abs(all_residuals),
           alpha=0.6, s=50, color='purple', edgecolors='black', linewidth=0.4)
ax.plot(trend_x_het, z_het[0]*trend_x_het+z_het[1],
        'r--', lw=2, label='Trend Line')
ax.set_xlabel('Predicted KQ', fontsize=11, fontweight='bold')
ax.set_ylabel('|Residuals|',  fontsize=11, fontweight='bold')
ax.set_title(f'Heteroscedasticity Check\nBreusch-Pagan p={bp_p:.4f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[2, 2])
ax.axis('off')
summary = f"""
Statistical Test Results:

1. Normality Tests:
   Shapiro-Wilk:  p={shapiro_p:.4f}  {'✓' if shapiro_p>0.05 else '⚠'}
   K-S Test:      p={ks_p:.4f}  {'✓' if ks_p>0.05 else '⚠'}
   Jarque-Bera:   p={jb_p:.4f}  {'✓' if jb_p>0.05 else '⚠'}

2. Independence:
   Durbin-Watson: {dw_stat:.4f}  {'✓' if 1.5<dw_stat<2.5 else '⚠'}

3. Homoscedasticity:
   Breusch-Pagan: p={bp_p:.4f}  {'✓' if bp_p>0.05 else '⚠'}

4. Correlation:
   Pred vs Resid: r={corr_all:.4f}  {'✓' if abs(corr_all)<0.3 else '⚠'}

Model Performance:
   Train R²:   {train_r2:.4f}
   Test  R²:   {test_r2:.4f}
   Train RMSE: {train_rmse:.4f}
   Test  RMSE: {test_rmse:.4f}
   Test  MAE:  {test_mae:.4f}
   Test  MAPE: {test_mape:.2f}%

Passed: {passed}/7 ({passed/7*100:.0f}%)
"""
ax.text(0.05, 0.5, summary, fontsize=9, family='monospace', va='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('GP Matérn-0.5 Model — Comprehensive Residual Analysis\n'
             '(Gp_GP_Matern_0.5_K6_Comb17)',
             fontsize=14, fontweight='bold', y=0.998)
plt.savefig(os.path.join(OUTPUT_DIR, '1_comprehensive_residual_analysis.png'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ 图1 保存: 1_comprehensive_residual_analysis.png")

# ==============================================================
# 图2：正态性检验详细（2×2）
# ==============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax = axes[0, 0]
stats.probplot(train_res_std, dist='norm', plot=ax)
ax.set_title(f'Training Set Q-Q Plot\nShapiro-Wilk p={shapiro(train_residuals)[1]:.4f}',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
stats.probplot(test_res_std, dist='norm', plot=ax)
ax.set_title(f'Test Set Q-Q Plot\nShapiro-Wilk p={shapiro(test_residuals)[1]:.4f}',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.hist(train_residuals, bins=12, density=True, alpha=0.7,
        color='steelblue', edgecolor='black', label='Train Residuals')
x_r = np.linspace(train_residuals.min(), train_residuals.max(), 100)
ax.plot(x_r, gaussian_kde(train_residuals)(x_r), 'r-', lw=2, label='KDE')
ax.plot(x_r, stats.norm.pdf(x_r, train_residuals.mean(), train_residuals.std()),
        'g--', lw=2, label='Normal Fit')
ax.set_xlabel('Residuals', fontsize=11, fontweight='bold')
ax.set_ylabel('Density',   fontsize=11, fontweight='bold')
ax.set_title('Training Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.hist(test_residuals, bins=8, density=True, alpha=0.7,
        color='darkorange', edgecolor='black', label='Test Residuals')
x_r2 = np.linspace(test_residuals.min(), test_residuals.max(), 100)
ax.plot(x_r2, gaussian_kde(test_residuals)(x_r2), 'r-', lw=2, label='KDE')
ax.plot(x_r2, stats.norm.pdf(x_r2, test_residuals.mean(), test_residuals.std()),
        'g--', lw=2, label='Normal Fit')
ax.set_xlabel('Residuals', fontsize=11, fontweight='bold')
ax.set_ylabel('Density',   fontsize=11, fontweight='bold')
ax.set_title('Test Set Residuals Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('Normality Tests — Detailed Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '2_normality_tests_detailed.png'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ 图2 保存: 2_normality_tests_detailed.png")

# ==============================================================
# 图3：独立性与随机性检验（2×2）
# ==============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax = axes[0, 0]
if max_lags_acf > 0:
    plot_acf(all_residuals, lags=max_lags_acf, ax=ax, alpha=0.05)
    ax.set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center')
    ax.axis('off')

ax = axes[0, 1]
if max_lags_pacf > 0:
    try:
        plot_pacf(all_residuals, lags=max_lags_pacf, ax=ax, alpha=0.05)
        ax.set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
    except Exception:
        ax.text(0.5, 0.5, 'PACF failed', ha='center', va='center'); ax.axis('off')
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center'); ax.axis('off')

ax = axes[1, 0]
if window > 0:
    ax.plot(all_residuals, 'o-', alpha=0.5, label='Residuals', markersize=4, color='teal')
    ax.plot(roll_mean.values, 'r-', lw=2, label=f'{window}-pt Moving Avg')
    ax.fill_between(range(len(all_residuals)),
                    (roll_mean-roll_std).values, (roll_mean+roll_std).values,
                    alpha=0.3, color='red')
    ax.axhline(0, color='black', ls='--', lw=1)
    ax.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
    ax.set_ylabel('Residuals',    fontsize=11, fontweight='bold')
    ax.set_title('Residuals with Moving Average', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Sample size too small', ha='center', va='center'); ax.axis('off')

ax = axes[1, 1]
ax.plot(np.cumsum(all_residuals), 'b-', lw=2)
ax.axhline(0, color='red', ls='--', lw=2)
ax.set_xlabel('Sample Index',                fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Sum of Residuals', fontsize=11, fontweight='bold')
ax.set_title(f'CUSUM Chart (Systematic Bias Check)\nDurbin-Watson={dw_stat:.4f}',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Independence & Randomness Tests', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '3_independence_and_randomness.png'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ 图3 保存: 3_independence_and_randomness.png")

# ==============================================================
# 图4：GP专属预测不确定性分析（2×2）
# ==============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax = axes[0, 0]
sc = ax.scatter(all_pred, all_std,
                c=np.abs(all_residuals), cmap='RdYlGn_r',
                alpha=0.7, s=60, edgecolors='black', linewidth=0.4)
plt.colorbar(sc, ax=ax, label='|Residuals|')
ax.set_xlabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=11, fontweight='bold')
ax.set_ylabel('GP Prediction Std (σ)',          fontsize=11, fontweight='bold')
ax.set_title('GP Uncertainty vs Predicted Value\n(Color = |Residual|)',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.scatter(all_std, np.abs(all_residuals),
           alpha=0.6, s=60, color='purple', edgecolors='black', linewidth=0.4)
max_val = max(all_std.max(), np.abs(all_residuals).max())
ax.plot([0,max_val], [0,max_val],   'r--', lw=2,   label='y = x (ideal)')
ax.plot([0,max_val], [0,2*max_val], 'orange', lw=1.5, ls=':', label='y = 2x')
ax.set_xlabel('GP Prediction Std (σ)', fontsize=11, fontweight='bold')
ax.set_ylabel('|Residuals|',           fontsize=11, fontweight='bold')
ax.set_title('GP Calibration Check\n(|Residual| vs Predicted σ)',
             fontsize=12, fontweight='bold')
ax.text(0.05, 0.95, f'|res|≤1σ: {within_1:.1f}%\n|res|≤2σ: {within_2:.1f}%',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8), va='top')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.hist(norm_resid_gp, bins=15, density=True, alpha=0.7,
        color='lightcoral', edgecolor='black', label='Normalized Residuals')
x_n2 = np.linspace(norm_resid_gp.min(), norm_resid_gp.max(), 100)
ax.plot(x_n2, stats.norm.pdf(x_n2, 0, 1),       'b-',  lw=2, label='N(0,1)')
ax.plot(x_n2, gaussian_kde(norm_resid_gp)(x_n2), 'r--', lw=2, label='KDE')
ax.set_xlabel('Normalized Residuals (res/σ_GP)', fontsize=11, fontweight='bold')
ax.set_ylabel('Density',                          fontsize=11, fontweight='bold')
ax.set_title('GP Normalized Residuals Distribution\n(Should Follow N(0,1) if Well-Calibrated)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
sort_idx = np.argsort(all_pred); x_plot = range(len(sort_idx))
ax.errorbar(x_plot, all_pred[sort_idx], yerr=2*all_std[sort_idx],
            fmt='none', alpha=0.3, color='steelblue',
            elinewidth=1, capsize=0, label='±2σ GP')
ax.scatter(x_plot, all_pred[sort_idx], s=20, color='steelblue',
           alpha=0.7, zorder=3, label='Predicted')
ax.scatter(x_plot, y_all[sort_idx], s=15, color='red',
           alpha=0.7, zorder=4, marker='x', label='Actual')
ax.set_xlabel('Sample (sorted by predicted KQ)', fontsize=11, fontweight='bold')
ax.set_ylabel('KQ (MPa·m$^{0.5}$)',              fontsize=11, fontweight='bold')
ax.set_title('Predicted ±2σ vs Actual KQ\n(GP Uncertainty Band)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('GP Model Exclusive: Prediction Uncertainty Analysis\n'
             '(GP Matérn-0.5 — return_std=True)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '4_GP_uncertainty_analysis.png'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ 图4 保存: 4_GP_uncertainty_analysis.png")

# ==============================================================
# 图5：Q-Q图增强完整版（2×3，6个子图）
# ==============================================================
print("\n  生成图5 Q-Q图增强完整版...")
fig, axes = plt.subplots(2, 3, figsize=(21, 14))

# 上行：带置信带的Q-Q图（训练 / 测试 / 全数据）
print("    绘制置信带Q-Q图（训练集）...")
draw_qq_with_ci(axes[0, 0], train_residuals,
                'Training Set Q-Q Plot\n(with 95% Confidence Band)',
                color='steelblue', ci=0.95, n_boot=N_BOOT_QQ)

print("    绘制置信带Q-Q图（测试集）...")
draw_qq_with_ci(axes[0, 1], test_residuals,
                'Test Set Q-Q Plot\n(with 95% Confidence Band)',
                color='darkorange', ci=0.95, n_boot=N_BOOT_QQ)

print("    绘制置信带Q-Q图（全数据）...")
draw_qq_with_ci(axes[0, 2], all_residuals,
                'All Data Q-Q Plot\n(with 95% Confidence Band)',
                color='seagreen', ci=0.95, n_boot=N_BOOT_QQ)

# 下行：标准化残差Q-Q图（训练 / 测试 / 全数据）
print("    绘制标准化残差Q-Q图...")
draw_standardized_qq(axes[1, 0], train_residuals,
                     'Training Set — Standardized Residuals Q-Q\n(vs N(0,1))',
                     color='steelblue')

draw_standardized_qq(axes[1, 1], test_residuals,
                     'Test Set — Standardized Residuals Q-Q\n(vs N(0,1))',
                     color='darkorange')

draw_standardized_qq(axes[1, 2], all_residuals,
                     'All Data — Standardized Residuals Q-Q\n(vs N(0,1))',
                     color='seagreen')

# 行标注
fig.text(0.01, 0.73, 'Q-Q with\n95% CI',
         fontsize=12, fontweight='bold', color='#333333',
         ha='left', va='center',
         bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow',
                   ec='gray', alpha=0.8))
fig.text(0.01, 0.27, 'Standardized\nResiduals',
         fontsize=12, fontweight='bold', color='#333333',
         ha='left', va='center',
         bbox=dict(boxstyle='round,pad=0.4', fc='lightcyan',
                   ec='gray', alpha=0.8))

plt.suptitle(
    'Q-Q Plot Enhanced Analysis — GP Matérn-0.5\n'
    'Top row: Q-Q with Bootstrap 95% Confidence Band  |  '
    'Bottom row: Standardized Residuals vs N(0,1)',
    fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, '5_QQ_enhanced_analysis.png'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ 图5 保存: 5_QQ_enhanced_analysis.png")

# ============================================================
# 文字报告
# ============================================================
report_path = os.path.join(OUTPUT_DIR, 'GP_Matern_Residual_Analysis_Report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("TOP1 高斯过程模型完整残差分析报告 v2（修复版）\n")
    f.write("修复: qq_confidence_band 返回值解包对齐（7个变量）\n")
    f.write("Model: Gp_GP_Matern_0.5_K6_Comb17\n")
    f.write("=" * 80 + "\n\n")
    f.write("1. 模型信息\n" + "-"*60 + "\n")
    f.write(f"  模型类型: GP Matérn-0.5\n")
    f.write(f"  特征（{len(feature_names)}个）: {', '.join(feature_names)}\n\n")
    f.write("2. 数据概况\n" + "-"*60 + "\n")
    f.write(f"  总样本: {len(y_all)}  训练: {len(y_train)}  测试: {len(y_test)}\n\n")
    f.write("3. 模型性能\n" + "-"*60 + "\n")
    f.write(f"  训练: R²={train_r2:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}\n")
    f.write(f"  测试: R²={test_r2:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}\n\n")
    f.write("4. 统计检验\n" + "-"*60 + "\n")
    f.write(f"  Shapiro-Wilk:   p={shapiro_p:.4f}  {'✓ Pass' if shapiro_p>0.05 else '⚠ Attention'}\n")
    f.write(f"  K-S Test:       p={ks_p:.4f}  {'✓ Pass' if ks_p>0.05 else '⚠ Attention'}\n")
    f.write(f"  Jarque-Bera:    p={jb_p:.4f}  {'✓ Pass' if jb_p>0.05 else '⚠ Attention'}\n")
    f.write(f"  Durbin-Watson:  {dw_stat:.4f}  {'✓ Pass' if 1.5<dw_stat<2.5 else '⚠ Attention'}\n")
    f.write(f"  Breusch-Pagan:  p={bp_p:.4f}  {'✓ Pass' if bp_p>0.05 else '⚠ Attention'}\n")
    f.write(f"  通过检验: {passed}/7 ({passed/7*100:.1f}%)\n\n")
    f.write("5. Q-Q图增强分析（v2新增）\n" + "-"*60 + "\n")
    f.write(f"  训练集 Shapiro-Wilk: p={sw_p_tr:.4f}  "
            f"{'✓ 正态' if sw_p_tr>0.05 else '⚠ 非正态'}\n")
    f.write(f"  测试集 Shapiro-Wilk: p={sw_p_te:.4f}  "
            f"{'✓ 正态' if sw_p_te>0.05 else '⚠ 非正态'}\n")
    f.write(f"  全数据 Shapiro-Wilk: p={sw_p_all:.4f}  "
            f"{'✓ 正态' if sw_p_all>0.05 else '⚠ 非正态'}\n")
    f.write(f"  Bootstrap置信带:     95% ({N_BOOT_QQ}次)\n\n")
    f.write("6. 输出文件\n" + "-"*60 + "\n")
    f.write("  GP_Matern_Residual_Analysis_Data.xlsx (29个工作表)\n")
    f.write("    Sheet 27: QQ_All_Data         → 全数据Q-Q原始数据\n")
    f.write("    Sheet 28: QQ_Confidence_Band  → 置信带数据（训练/测试/全）\n")
    f.write("    Sheet 29: QQ_Standardized     → 标准化残差Q-Q数据\n")
    f.write("  1_comprehensive_residual_analysis.png\n")
    f.write("  2_normality_tests_detailed.png\n")
    f.write("  3_independence_and_randomness.png\n")
    f.write("  4_GP_uncertainty_analysis.png\n")
    f.write("  5_QQ_enhanced_analysis.png  ← v2新增\n")
    f.write(f"\n报告生成时间: {pd.Timestamp.now()}\n")
    f.write("=" * 80 + "\n")

print(f"  ✓ 文字报告已保存")

print("\n" + "="*80)
print("✅ TOP1 GP 模型完整残差分析 v2（修复版）完成！")
print("="*80)
print(f"\n📁 输出目录: {OUTPUT_DIR}")
print(f"\n📊 输出文件汇总:")
print(f"  • GP_Matern_Residual_Analysis_Data.xlsx  (29个工作表)")
print(f"    └─ Sheet27: QQ_All_Data（含slope/intercept）")
print(f"    └─ Sheet28: QQ_Confidence_Band（宽表，可直接Origin绘带状图）")
print(f"    └─ Sheet29: QQ_Standardized（标准化残差Q-Q数据）")
print(f"  • 1_comprehensive_residual_analysis.png")
print(f"  • 2_normality_tests_detailed.png")
print(f"  • 3_independence_and_randomness.png")
print(f"  • 4_GP_uncertainty_analysis.png")
print(f"  • 5_QQ_enhanced_analysis.png             ★ v2新增")
print(f"    └─ 上行: 训练/测试/全数据 Q-Q + 95%Bootstrap置信带")
print(f"    └─ 下行: 训练/测试/全数据 标准化残差Q-Q vs N(0,1)")
print(f"\n✨ 统计检验通过率: {passed}/7 ({passed/7*100:.0f}%)")
print(f"✨ 模型性能: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")
print(f"✨ Q-Q SW p值: 训练={sw_p_tr:.4f}  测试={sw_p_te:.4f}  全={sw_p_all:.4f}")
print("="*80)

TOP1 高斯过程模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整残差分析程序 v2（修复版）
修复: qq_confidence_band 返回值解包对齐（7个变量）

[1/5] 加载 TOP1 高斯过程模型和数据...
  ✓ 模型名称: GP_Matern_0.5
  ✓ 特征列表: ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
  ✓ 原始 Test R²:  0.759772415859425
  ✓ 原始 Test MAE: 1.2079988124886822
  ✓ 原始数据: (202, 209)  有效样本: 202
  ✓ 目标变量范围: [3.400, 19.800]

[2/5] 数据划分...
  ✓ 训练集: 161 样本  测试集: 41 样本

[3/5] 模型推理...
  训练集: R²=0.8748, RMSE=1.0899, MAE=0.8392
  测试集: R²=0.7598, RMSE=1.5575, MAE=1.2080

[4/5] 残差分析与统计检验...
  Shapiro-Wilk: p=0.0512  ✓
  K-S Test:     p=0.5301  ✓
  Durbin-Watson: 1.5227  ✓
  Breusch-Pagan: p=0.0501  ✓
  通过检验: 5/7 (71.4%)

  ★ 计算Q-Q置信带数据（Bootstrap n=500）...
  ✓ 置信带计算完成（训练: n=161, 测试: n=41, 全: n=202）

  保存 Excel 数据文件（29个工作表）...
  ✓ Excel 已保存（29个工作表）: GP_Matern_Residual_Analysis_Data.xlsx

[5/5] 生成可视化图片（原4张 + 新增图5）...
  ✓ 图1 保存: 1_comprehensive_residual_analysis.png
  ✓ 图2 保存: 2_normality_tests_detailed.png
  ✓ 图3 保存: 3_independence_and_randomness.png
  ✓ 图4